### Routine to train and tabnet model

##### TODOs:
- Set MlFlow tracking URI
- Start mlflow server: mlflow server --host 127.0.0.1 --port 8080 (LOCAL)
- Change folders if needed

In [2]:
# https://github.com/dreamquark-ai/tabnet/blob/develop/pytorch_tabnet/tab_network.py

In [27]:
import os
import json
from datetime import datetime, timedelta
from pathlib import Path
from typing import Tuple, Optional, Union, List, Dict
from dataclasses import dataclass, field
import multiprocessing
import numpy as np
import pandas as pd
from shapely.geometry import Point
import seaborn as sns
from scipy.stats import skew, kurtosis, entropy, randint, uniform, loguniform
from scipy.fft import fft
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.calibration import calibration_curve
import xgboost as xgb
from xgboost import plot_importance
import joblib
from joblib import Parallel, delayed
import pyarrow as pa
from tqdm import tqdm
import mlflow
from mlflow.models.signature import infer_signature
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc,
    precision_recall_curve, average_precision_score, confusion_matrix, classification_report
)
from sklearn.metrics import roc_auc_score, roc_curve
# Deep learning and specialized ML libraries
import torch
from pytorch_tabnet.tab_model import TabNetClassifier
from pytorch_tabnet.pretraining import TabNetPretrainer


import warnings

warnings.filterwarnings('ignore')

In [4]:
#! mlflow server --host 127.0.0.1 --port 8080

In [5]:
# Set MlFlow tracking URI
mlflow.set_tracking_uri("http://localhost:8080") # Check your MLflow server URI

In [6]:
MODEL_NAME = "geok_tabnet_no_q_new_enc_dim_20"
DATE = datetime.now().strftime("%Y%m%d_%H%M%S")

In [7]:
class DDMFeatureExtractor:
    def __init__(self):
        pass
    @staticmethod
    def gini(array):
            """Gini coefficient calculation"""
            array = np.sort(array)
            index = np.arange(1, array.shape[0] + 1)
            return (np.sum((2 * index - array.shape[0] - 1) * array)) / (array.shape[0] * np.sum(array))  
      
    def extract_ddm_features(self, fit_data: np.ndarray) -> pd.DataFrame:
        """
        Extract features from DDM data.
        """
        features = []

        for row in tqdm(fit_data, desc="Extracting DDM features"):
            f = {}
            x = np.array(row, dtype=np.float64) + 1e-10  # evita log(0)

            # 1. General statistics
            f['mean'] = np.mean(x)
            f['std'] = np.std(x)
            f['min'] = np.min(x)
            f['max'] = np.max(x)
            f['median'] = np.median(x)
            f['range'] = np.max(x) - np.min(x)
            f['skew'] = skew(x)
            f['kurtosis'] = kurtosis(x)
            f['entropy'] = entropy(x)
            f['gini'] = self.gini(x)

            # 2. Positional 
            f['peak_index'] = np.argmax(x)
            f['peak_value'] = np.max(x)
            f['center_of_mass'] = np.sum(np.arange(len(x)) * x) / np.sum(x)
            f['inertia'] = np.sum(((np.arange(len(x)) - f['center_of_mass'])**2) * x)

            # 3. Segmentations in thirds
            thirds = np.array_split(x, 3)
            for i, part in enumerate(thirds):
                f[f'sum_third_{i+1}'] = np.sum(part)
                f[f'mean_third_{i+1}'] = np.mean(part)
                f[f'max_third_{i+1}'] = np.max(part)

            # 3.1 Segmentations in windows of 5
            windows = np.array_split(x, 5)
            for i, w in enumerate(windows):
                f[f'mean_w{i+1}'] = np.mean(w)
                f[f'std_w{i+1}'] = np.std(w)
                f[f'max_w{i+1}'] = np.max(w)

            # 4. Derivative statistics and differences
            dx = np.diff(x)
            f['mean_diff'] = np.mean(dx)
            f['std_diff'] = np.std(dx)
            f['max_diff'] = np.max(dx)
            f['min_diff'] = np.min(dx)
            f['n_positive_diff'] = np.sum(dx > 0)
            f['n_negative_diff'] = np.sum(dx < 0)
            f['n_zero_diff'] = np.sum(dx == 0)

            # 5. Autocorrelations (lag 1-3)
            for lag in range(1, 4):
                ac = np.corrcoef(x[:-lag], x[lag:])[0, 1] if len(x) > lag else np.nan
                f[f'autocorr_lag{lag}'] = ac

            # 6. FFT 
            spectrum = np.abs(fft(x)) # type: ignore
            half_spectrum = spectrum[:len(spectrum)//2]  
            f['fft_peak_freq'] = np.argmax(half_spectrum)
            f['fft_max'] = np.max(half_spectrum)
            f['fft_median'] = np.median(half_spectrum)
            f['fft_mean'] = np.mean(half_spectrum)


            features.append(f)
        return features # type: ignore

In [ ]:
def combined_features_to_dataframe(combined_features, full_data = pd.DataFrame(), full_labels = pd.DataFrame()) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
    flat_features = [row[0] if isinstance(row, list) and len(row) > 0 else row for row in combined_features]
    FEATURES=list(combined_features[0][0].keys())
    combined_features = np.array([[row[key] for key in FEATURES] for row in flat_features])
    del flat_features
    combined_features.shape

    # Filter out rows with NaN or infinite values
    mask_finite = np.isfinite(combined_features).all(axis=1) & (np.abs(combined_features) < np.finfo(np.float64).max).all(axis=1)
    fit_data_with_features_clean = combined_features[mask_finite]
    labels_clean = full_labels[mask_finite]
    del combined_features
    return pd.DataFrame(fit_data_with_features_clean, columns=FEATURES),  pd.DataFrame(labels_clean, columns=['0']), FEATURES

In [ ]:
# Load data from Parquet
parquet_path = r"E:/data/balanced_df_enh_encoder_5M.parquet"
df_parquet = pd.read_parquet(parquet_path)
full_data = df_parquet.drop(columns=['label']).values
full_labels = df_parquet['label'].values

#Load test data from Parquet
parquet_path_test = r"E:/data/test_df_enh_encoder_1M.parquet"
df_parquet_test = pd.read_parquet(parquet_path_test)
full_data_test = df_parquet_test.drop(columns=['label']).values
full_labels_test = df_parquet_test['label'].values

In [11]:
# Extract DDM features in parallel

cpu_cores = multiprocessing.cpu_count()
print(f"Available {cpu_cores} CPU cores for parallel processing.")

Available 16 CPU cores for parallel processing.


In [12]:
import time
#time.sleep(60*60*5)

In [18]:
# Create a stratified subset (e.g., 10% of the data)
X_subset, _, y_subset, _ = train_test_split(
    pd.DataFrame(full_data),
    pd.DataFrame(full_labels),
    test_size=0.05,
    stratify=full_labels,
    random_state=42
)

# Reset index for convenience
X_subset = X_subset.reset_index(drop=True)
y_subset = y_subset.reset_index(drop=True)
len(X_subset), len(y_subset)

(4750000, 4750000)

In [19]:
FEATURES = [str(i) for i in range(20)]
len(FEATURES)

20

In [20]:
X_subset.columns = FEATURES
y_subset.columns = ['0']

In [28]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_recall_curve, roc_curve, f1_score, 
    precision_score, recall_score, accuracy_score,
    roc_auc_score, classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

class TabNetBinaryClassifier:
    """
    Class for training a TabNet binary classifier with GPU support and threshold optimization
    """
    
    def __init__(self, 
                 X_original,
                 y_original,
                 scaler_path,
                 n_d, 
                 n_a, 
                 n_steps, 
                 gamma,
                 n_independent=2,
                 n_shared=2,
                 lambda_sparse=1e-3,
                 optimizer_fn=torch.optim.Adam,
                 optimizer_params=dict(lr=1e-2),
                 mask_type='entmax',
                 momentum=0.02,
                 clip_value=1,
                 scheduler_params=dict(step_size=50, gamma=0.9),
                 scheduler_fn=torch.optim.lr_scheduler.StepLR,
                 epsilon=1e-15,
                 device_name='auto'):
        """
        Initialize the TabNet classifier
        """
        
        # Device configuration
        self.device = self._setup_device(device_name)
        print(f"Device used: {self.device}")
        
        self.tabnet_params = {
            'n_d': n_d,
            'n_a': n_a, 
            'n_steps': n_steps,
            'gamma': gamma,
            'n_independent': n_independent,
            'n_shared': n_shared,
            'lambda_sparse': lambda_sparse,
            'optimizer_fn': optimizer_fn,
            'optimizer_params': optimizer_params,
            'mask_type': mask_type,
            'scheduler_params': scheduler_params,
            'scheduler_fn': scheduler_fn,
            'epsilon': epsilon,
            'device_name': self.device
        }
        self.X_original = X_original
        self.y_original = y_original
        self.model = None
        self.scaler = StandardScaler()
        self.feature_names = None
        self.is_fitted = False
        self.scaler_path = scaler_path
        
        # Threshold optimization attributes
        self.optimal_threshold = 0.5
        self.threshold_metrics = {}
        
    def _setup_device(self, device_name):
        """Configure the computing device (CPU/GPU)"""
        if device_name == 'auto':
            if torch.cuda.is_available():
                device = 'cuda'
                print(f"GPU available: {torch.cuda.get_device_name()}")
                print(f"GPU memory available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
            else:
                device = 'cpu'
                print("GPU not available, using CPU")
        else:
            device = device_name
            if device.startswith('cuda') and not torch.cuda.is_available():
                print("WARNING: GPU requested but not available, using CPU")
                device = 'cpu'
        
        return device
    
    def get_gpu_memory_info(self):
        """Returns GPU memory information"""
        if torch.cuda.is_available() and self.device.startswith('cuda'):
            device_idx = 0 if self.device == 'cuda' else int(self.device.split(':')[1])
            allocated = torch.cuda.memory_allocated(device_idx) / 1e9
            reserved = torch.cuda.memory_reserved(device_idx) / 1e9
            total = torch.cuda.get_device_properties(device_idx).total_memory / 1e9
            
            print(f"GPU Memory:")
            print(f"  - Allocated: {allocated:.2f} GB")
            print(f"  - Reserved: {reserved:.2f} GB") 
            print(f"  - Total: {total:.2f} GB")
            print(f"  - Free: {total - reserved:.2f} GB")
            
            return {
                'allocated': allocated,
                'reserved': reserved,
                'total': total,
                'free': total - reserved
            }
        else:
            print("GPU memory not available")
            return None
    
    def clear_gpu_memory(self):
        """Clear GPU memory"""
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            print("GPU cache cleared")
    
    def prepare_data(self, X, y, test_size=0.2, random_state=42):
        """Prepare data for training"""
        
        # Separate features and target
        X = self.X_original
        y = self.y_original

        # Save feature names
        self.feature_names = X.columns.tolist()
        
        # Convert to float32 to optimize GPU memory
        X = X.astype(np.float32)
        
        # Split X and y into train (64%), validation (16%), and test (20%) sets
        X_temp, X_test, y_temp, y_test = train_test_split(
            X,
            y,
            test_size=test_size,
            stratify=y,
            random_state=42
        )

        X_train, X_val, y_train, y_val = train_test_split(
            X_temp,
            y_temp,
            test_size=test_size,  # 0.2 * 0.8 = 0.16 of the original data
            stratify=y_temp,
            random_state=42
        )

        # Reset indices for convenience
        X_train = X_train.reset_index(drop=True)
        X_val = X_val.reset_index(drop=True)
        X_test = X_test.reset_index(drop=True)
        y_train = y_train.reset_index(drop=True)
        y_val = y_val.reset_index(drop=True)
        y_test = y_test.reset_index(drop=True)

        # Numeric columns will be scaled by StandardScaler
        # Load scaler from path if provided
        if self.scaler_path is not None:
            scaler = joblib.load(self.scaler_path)
            print(f"Scaler loaded from: {self.scaler_path}")
        else:
            scaler = StandardScaler()

        column_trans = ColumnTransformer(
            [ ('scaler',scaler, FEATURES),
            ], remainder='passthrough', n_jobs=-1)

        train_X_transformed = column_trans.fit_transform(X_train, y_train)
        val_X_transformed = column_trans.transform(X_val )
        test_X_transformed = column_trans.transform(X_test)

        self.X_train = train_X_transformed
        self.X_val = val_X_transformed
        self.X_test = test_X_transformed

        # Convert to float32 for GPU
        self.X_train = X_train.values.astype(np.float32)
        self.y_train = y_train.values.astype(np.int64)

        self.X_test = X_test.values.astype(np.float32)
        self.y_test = y_test.values.astype(np.int64)

        self.X_val = X_val.values.astype(np.float32)
        self.y_val = y_val.values.astype(np.int64)

        print(f"Data prepared:")
        print(f"  - Training set: {self.X_train.shape}")
        print(f"  - Test set: {self.X_test.shape}")
        print(f"  - Validation set: {self.X_val.shape}")

        return self.X_train, self.X_test, self.y_train, self.y_test, self.X_val, self.y_val
    
    def train(self, 
              max_epochs=200, 
              patience=15, 
              batch_size=1024,
              virtual_batch_size=128,
              num_workers=0,
              drop_last=False):
        """Train the TabNet model"""
        
        self.X_train, self.X_test, self.y_train, self.y_test, self.X_val, self.y_val = self.prepare_data(self.X_original, self.y_original, test_size=0.2, random_state=42)
        
        # Adapt batch_size for GPU
        if self.device.startswith('cuda'):
            gpu_memory = self.get_gpu_memory_info()
            if gpu_memory and gpu_memory['free'] < 2.0:  # Less than 2GB free
                suggested_batch_size = min(batch_size, 512)
                print(f"Limited GPU memory, reducing batch_size to {suggested_batch_size}")
                batch_size = suggested_batch_size
            
            # Optimize num_workers for GPU
            if num_workers == 0:
                num_workers = min(4, torch.cuda.device_count() * 2)
                
        print("Training configuration:")
        print(f"  - Device: {self.device}")
        print(f"  - Batch size: {batch_size}")
        print(f"  - Virtual batch size: {virtual_batch_size}")
        print(f"  - Num workers: {num_workers}")
        
        # Initialize model
        self.model = TabNetClassifier(**self.tabnet_params)
        
        # Check memory before training
        if self.device.startswith('cuda'):
            self.clear_gpu_memory()
            print("GPU memory before training:")
            self.get_gpu_memory_info()
        
        # Training
        print("\nStarting TabNet training...") 
        
        try:
            self.model.fit(
                X_train=self.X_train,
                y_train=self.y_train.reshape(-1),
                eval_set=[(self.X_val, self.y_val.reshape(-1))],
                eval_name=['test'],
                eval_metric=['accuracy', 'auc'],
                max_epochs=100,
                patience=patience,
                batch_size=512,
                virtual_batch_size=256,
                num_workers=num_workers,
                drop_last=drop_last,
            )
            
            self.is_fitted = True
            print("Training completed!")
            
            # Check memory after training
            if self.device.startswith('cuda'):
                print("\nGPU memory after training:")
                self.get_gpu_memory_info()
                
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print("\nERROR: Insufficient GPU memory!")
                self.clear_gpu_memory()
            raise e
        
        return self.model
    
    def optimize_threshold(self, X=None, y=None, metric='f1', plot=True):
        """
        Optimize classification threshold based on validation set
        
        Parameters:
        -----------
        X : array-like, optional
            Features for threshold optimization. If None, uses validation set
        y : array-like, optional
            True labels. If None, uses validation set labels
        metric : str
            Metric to optimize ('f1', 'precision', 'recall', 'accuracy', 'youden')
        plot : bool
            Whether to plot threshold optimization curve
            
        Returns:
        --------
        dict : Dictionary with optimal threshold and metrics
        """
        
        if not self.is_fitted:
            raise ValueError("Model has not been trained yet!")
        
        # Use validation set if no data provided
        if X is None:
            X = self.X_val
            y = self.y_val.ravel()
        
        # Get prediction probabilities
        y_pred_proba = self.predict_proba(X)[:, 1]
        
        # Define threshold range
        thresholds = np.arange(0.1, 0.95, 0.01)
        
        # Calculate metrics for each threshold
        metrics_data = {
            'threshold': [],
            'precision': [],
            'recall': [],
            'f1': [],
            'accuracy': [],
            'youden': []  # Youden's J statistic (Sensitivity + Specificity - 1)
        }
        
        for threshold in thresholds:
            y_pred = (y_pred_proba >= threshold).astype(int)
            
            precision = precision_score(y, y_pred, zero_division=0)
            recall = recall_score(y, y_pred, zero_division=0)
            f1 = f1_score(y, y_pred, zero_division=0)
            accuracy = accuracy_score(y, y_pred)
            
            # Calculate Youden's J statistic
            tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
            sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
            youden = sensitivity + specificity - 1
            
            metrics_data['threshold'].append(threshold)
            metrics_data['precision'].append(precision)
            metrics_data['recall'].append(recall)
            metrics_data['f1'].append(f1)
            metrics_data['accuracy'].append(accuracy)
            metrics_data['youden'].append(youden)
        
        # Convert to DataFrame for easier manipulation
        metrics_df = pd.DataFrame(metrics_data)
        
        # Find optimal threshold based on selected metric
        if metric == 'youden':
            optimal_idx = metrics_df['youden'].idxmax()
        else:
            optimal_idx = metrics_df[metric].idxmax()
        
        self.optimal_threshold = metrics_df.loc[optimal_idx, 'threshold']
        self.threshold_metrics = metrics_df.loc[optimal_idx].to_dict()
        
        print(f"\n=== THRESHOLD OPTIMIZATION RESULTS ===")
        print(f"Optimization metric: {metric}")
        print(f"Optimal threshold: {self.optimal_threshold:.3f}")
        print(f"Metrics at optimal threshold:")
        print(f"  - Precision: {self.threshold_metrics['precision']:.4f}")
        print(f"  - Recall: {self.threshold_metrics['recall']:.4f}")
        print(f"  - F1-Score: {self.threshold_metrics['f1']:.4f}")
        print(f"  - Accuracy: {self.threshold_metrics['accuracy']:.4f}")
        print(f"  - Youden's J: {self.threshold_metrics['youden']:.4f}")
        
        if plot:
            self.plot_threshold_optimization(metrics_df, metric)
        
        return {
            'optimal_threshold': self.optimal_threshold,
            'metrics': self.threshold_metrics,
            'all_metrics': metrics_df
        }
    
    def plot_threshold_optimization(self, metrics_df, optimization_metric):
        """
        Plot threshold optimization curves
        
        Parameters:
        -----------
        metrics_df : pandas.DataFrame
            DataFrame with metrics for different thresholds
        optimization_metric : str
            The metric used for optimization
        """
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Plot 1: All metrics vs threshold
        axes[0, 0].plot(metrics_df['threshold'], metrics_df['precision'], 
                       label='Precision', linewidth=2)
        axes[0, 0].plot(metrics_df['threshold'], metrics_df['recall'], 
                       label='Recall', linewidth=2)
        axes[0, 0].plot(metrics_df['threshold'], metrics_df['f1'], 
                       label='F1-Score', linewidth=2)
        axes[0, 0].plot(metrics_df['threshold'], metrics_df['accuracy'], 
                       label='Accuracy', linewidth=2)
        
        # Highlight optimal threshold
        axes[0, 0].axvline(x=self.optimal_threshold, color='red', 
                          linestyle='--', alpha=0.7, 
                          label=f'Optimal ({optimization_metric})')
        
        axes[0, 0].set_xlabel('Threshold')
        axes[0, 0].set_ylabel('Metric Value')
        axes[0, 0].set_title('All Metrics vs Threshold')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        # Plot 2: Precision-Recall trade-off
        axes[0, 1].plot(metrics_df['threshold'], metrics_df['precision'], 
                       label='Precision', linewidth=2, color='blue')
        axes[0, 1].plot(metrics_df['threshold'], metrics_df['recall'], 
                       label='Recall', linewidth=2, color='orange')
        axes[0, 1].axvline(x=self.optimal_threshold, color='red', 
                          linestyle='--', alpha=0.7)
        axes[0, 1].set_xlabel('Threshold')
        axes[0, 1].set_ylabel('Score')
        axes[0, 1].set_title('Precision-Recall Trade-off')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # Plot 3: F1-Score and Youden's J
        axes[1, 0].plot(metrics_df['threshold'], metrics_df['f1'], 
                       label='F1-Score', linewidth=2, color='green')
        axes[1, 0].plot(metrics_df['threshold'], metrics_df['youden'], 
                       label="Youden's J", linewidth=2, color='purple')
        axes[1, 0].axvline(x=self.optimal_threshold, color='red', 
                          linestyle='--', alpha=0.7)
        axes[1, 0].set_xlabel('Threshold')
        axes[1, 0].set_ylabel('Score')
        axes[1, 0].set_title('F1-Score and Youden\'s J vs Threshold')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        # Plot 4: Metric distribution at optimal threshold
        optimal_metrics = [
            self.threshold_metrics['precision'],
            self.threshold_metrics['recall'],
            self.threshold_metrics['f1'],
            self.threshold_metrics['accuracy']
        ]
        metric_names = ['Precision', 'Recall', 'F1-Score', 'Accuracy']
        
        bars = axes[1, 1].bar(metric_names, optimal_metrics, 
                             color=['blue', 'orange', 'green', 'red'], alpha=0.7)
        axes[1, 1].set_ylabel('Score')
        axes[1, 1].set_title(f'Metrics at Optimal Threshold ({self.optimal_threshold:.3f})')
        axes[1, 1].set_ylim(0, 1)
        
        # Add value labels on bars
        for bar, value in zip(bars, optimal_metrics):
            axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                           f'{value:.3f}', ha='center', va='bottom')
        
        axes[1, 1].grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        plt.show()
    
    def predict(self, X=None, use_optimal_threshold=True):
        """
        Make predictions
        
        Parameters:
        -----------
        X : array-like, optional
            Features to predict. If None, uses test set
        use_optimal_threshold : bool
            Whether to use optimal threshold or default 0.5
        """
        if not self.is_fitted:
            raise ValueError("Model has not been trained yet!")
        
        if X is None:
            X = self.X_test
            
        # Convert to float32 for consistency
        if isinstance(X, pd.DataFrame):
            X = X.values.astype(np.float32)
        elif not isinstance(X, np.ndarray):
            X = np.array(X, dtype=np.float32)
        else:
            X = X.astype(np.float32)
        
        # Get probabilities
        probabilities = self.model.predict_proba(X)
        
        # Apply threshold
        threshold = self.optimal_threshold if use_optimal_threshold else 0.5
        predictions = (probabilities[:, 1] >= threshold).astype(int)
        
        return predictions
    
    def predict_proba(self, X=None):
        """Return prediction probabilities"""
        if not self.is_fitted:
            raise ValueError("Model has not been trained yet!")
        
        if X is None:
            X = self.X_test
            
        # Convert to float32 for consistency
        if isinstance(X, pd.DataFrame):
            X = X.values.astype(np.float32)
        elif not isinstance(X, np.ndarray):
            X = np.array(X, dtype=np.float32)
        else:
            X = X.astype(np.float32)
            
        probabilities = self.model.predict_proba(X)
        return probabilities
    
    def evaluate(self, X=None, y=None, use_optimal_threshold=True, plot_results=True):
        """
        Evaluate model performance
        
        Parameters:
        -----------
        X : array-like, optional
            Features to evaluate. If None, uses test set
        y : array-like, optional
            True labels. If None, uses test set labels
        use_optimal_threshold : bool
            Whether to use optimal threshold
        plot_results : bool
            Whether to plot results
        """
        if not self.is_fitted:
            raise ValueError("Model has not been trained yet!")
        
        if X is None:
            X = self.X_test
            y = self.y_test
        
        # Predictions
        y_pred = self.predict(X, use_optimal_threshold=use_optimal_threshold)
        y_pred_proba = self.predict_proba(X)
        
        # Metrics
        accuracy = accuracy_score(y, y_pred)
        auc_score = roc_auc_score(y, y_pred_proba[:, 1])
        precision = precision_score(y, y_pred)
        recall = recall_score(y, y_pred)
        f1 = f1_score(y, y_pred)
        
        threshold_used = self.optimal_threshold if use_optimal_threshold else 0.5
        
        print(f"\n=== EVALUATION RESULTS ===")
        print(f"Threshold used: {threshold_used:.3f}")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1-Score: {f1:.4f}")
        print(f"AUC Score: {auc_score:.4f}")
        print(f"\nClassification Report:")
        print(classification_report(y, y_pred))

        if plot_results:
            self.plot_results(y, y_pred, y_pred_proba[:, 1])
            
        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'auc_score': auc_score,
            'threshold_used': threshold_used,
            'predictions': y_pred,
            'probabilities': y_pred_proba
        }
    
    def plot_results(self, y_true, y_pred, y_pred_proba):
        """Visualize results"""
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        # Confusion Matrix
        cm = confusion_matrix(y_true, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
        axes[0].set_title('Confusion Matrix')
        axes[0].set_xlabel('Predicted')
        axes[0].set_ylabel('Actual')
        
        # ROC Curve
        fpr, tpr, _ = roc_curve(self.y_test.ravel(), y_pred_proba)
        auc = roc_auc_score(self.y_test.ravel(), y_pred_proba)
        
        axes[1].plot(fpr, tpr, label=f'ROC Curve (AUC = {auc:.3f})')
        axes[1].plot([0, 1], [0, 1], 'k--', label='Random')
        axes[1].set_xlabel('False Positive Rate')
        axes[1].set_ylabel('True Positive Rate')
        axes[1].set_title('ROC Curve')
        axes[1].legend()
        axes[1].grid(True)

        # Distribution of Probabilities
        axes[2].hist(y_pred_proba[self.y_test.ravel() == 0], bins=30, alpha=0.7, label='Class 0', color='red')
        axes[2].hist(y_pred_proba[self.y_test.ravel() == 1], bins=30, alpha=0.7, label='Class 1', color='blue')
        axes[2].axvline(x=self.optimal_threshold, color='green', linestyle='--', 
                       label=f'Optimal Threshold ({self.optimal_threshold:.3f})', linewidth=2)
        axes[2].set_xlabel('Predicted Probability')
        axes[2].set_ylabel('Frequency')
        axes[2].set_title('Distribution of Predicted Probabilities')
        axes[2].legend()
        axes[2].grid(True)
        
        plt.tight_layout()
        plt.show()
    
    def plot_feature_importance(self, plot=True, max_features=20):
        """Plot feature importance"""
        if self.model is not None and hasattr(self.model, "feature_importances_"):
            importances = self.model.feature_importances_
            indices = np.argsort(importances)[::-1][:20]
            plt.figure(figsize=(12, 6))
            plt.title("Top 20 Feature Importances")
            plt.bar(range(len(indices)), importances[indices], align="center")
            plt.xticks(range(len(indices)), [self.feature_names[i] for i in indices], rotation=90)
            plt.tight_layout()
            plt.show()
        else:
            print("Feature importances not available for this model.")

    def save_model(self, filepath):
        """Save the model"""
        if not self.is_fitted:
            raise ValueError("Model has not been trained yet!")
        
        # Save TabNet model
        self.model.save_model(filepath)
        
        # Save additional attributes (threshold, etc.)
        additional_data = {
            'optimal_threshold': self.optimal_threshold,
            'threshold_metrics': self.threshold_metrics,
            'feature_names': self.feature_names
        }
        
        import pickle
        with open(f"{filepath}_additional.pkl", 'wb') as f:
            pickle.dump(additional_data, f)
        
        print(f"Model and additional data saved at: {filepath}")
    
    def load_model(self, filepath):
        """Load a saved model"""
        self.model = TabNetClassifier(device_name=self.device)
        self.model.load_model(filepath)
        self.is_fitted = True
        
        # Load additional attributes
        try:
            import pickle
            with open(f"{filepath}_additional.pkl", 'rb') as f:
                additional_data = pickle.load(f)
            
            self.optimal_threshold = additional_data.get('optimal_threshold', 0.5)
            self.threshold_metrics = additional_data.get('threshold_metrics', {})
            self.feature_names = additional_data.get('feature_names', None)
            
        except FileNotFoundError:
            print("Additional data file not found, using defaults")
            self.optimal_threshold = 0.5
            self.threshold_metrics = {}
        
        print(f"Model loaded from: {filepath}")
        print(f"Optimal threshold: {self.optimal_threshold:.3f}")
    
    def get_model_summary(self):
        """Return model and hardware summary"""
        if not self.is_fitted:
            print("Model not yet trained")
            return
        
        print(f"\n=== MODEL SUMMARY ===")
        print(f"Device: {self.device}")
        print(f"Optimal threshold: {self.optimal_threshold:.3f}")
        print(f"TabNet Parameters:")
        for key, value in self.tabnet_params.items():
            if key != 'device_name':
                print(f"  - {key}: {value}")
        
        if hasattr(self.model, 'network'):
            total_params = sum(p.numel() for p in self.model.network.parameters())
            trainable_params = sum(p.numel() for p in self.model.network.parameters() if p.requires_grad)
            print(f"Total parameters: {total_params:,}")
            print(f"Trainable parameters: {trainable_params:,}")
        
        if self.device.startswith('cuda'):
            self.get_gpu_memory_info()


In [ ]:
# Inizializza il classificatore
classifier = TabNetBinaryClassifier(
    X_original=X_subset,
    y_original=y_subset,
    scaler_path=None,
    n_d=77,
    n_a=110,
    n_steps=8,
    optimizer_fn=torch.optim.AdamW,
    mask_type='entmax',
    n_independent=1,
    n_shared=5,
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    gamma=1.9,
    #clip_value=1,
    lambda_sparse=0.0005807365274596347,
    optimizer_params=dict(lr=0.003944458659837453),
    scheduler_params=dict(step_size=31, gamma=0.953213126517786),
    device_name='auto'  # Automatically detect GPU
)


In [ ]:
"""n_d 	77
n_a 	110
n_steps 	8
gamma 	1.9
cat_idxs 	[]
cat_dims 	[]
cat_emb_dim 	[]
n_independent 	1
n_shared 	5
epsilon 	1e-15
momentum 	0.02
lambda_sparse 	0.0005807365274596347
seed 	0
clip_value 	1
verbose 	1
optimizer_fn 	<class 'torch...im.adam.Adam'>
optimizer_params 	{'lr': 0.003944458659837453}
scheduler_fn 	<class 'torch...duler.StepLR'>
scheduler_params 	{'gamma': 0.953213126517786, 'step_size': 31}
mask_type 	'entmax'
input_dim 	52
output_dim 	2
device_name 	'cuda'
n_shared_decoder 	1
n_indep_decoder 	1
grouped_features 	[]"""

In [ ]:
# Addestra il modello
classifier.train(max_epochs=100, patience=25)

In [ ]:
# Evaluate performance
results = classifier.evaluate()
    
classifier.plot_feature_importance(max_features=20)

In [ ]:
# Ottimizza la soglia (puoi scegliere tra 'f1', 'precision', 'recall', 'accuracy', 'youden')
threshold_results = classifier.optimize_threshold(metric='youden', plot=True)

# Valuta con la soglia ottimizzata
results = classifier.evaluate(use_optimal_threshold=True, plot_results=True)

# Fai predizioni con la soglia ottimizzata
predictions = classifier.predict(use_optimal_threshold=True)

In [ ]:
# Ottimizza la soglia (puoi scegliere tra 'f1', 'precision', 'recall', 'accuracy', 'youden')
threshold_results = classifier.optimize_threshold(metric='accuracy', plot=True)

# Valuta con la soglia ottimizzata
results = classifier.evaluate(use_optimal_threshold=True, plot_results=True)

# Fai predizioni con la soglia ottimizzata
predictions = classifier.predict(use_optimal_threshold=True)

### Train tabnet

In [ ]:
"""=== OPTIMIZATION COMPLETED ===
Best auc: 0.9657
Best parameters:
  - n_d: 77
  - n_a: 110
  - n_steps: 8
  - gamma: 1.9067107319494947
  - n_independent: 1
  - n_shared: 5
  - lambda_sparse: 0.0005807365274596347
  - lr: 0.003944458659837453
  - step_size: 31
  - scheduler_gamma: 0.953213126517786
  - batch_size: 512
  - virtual_batch_size: 128
  """

In [ ]:
FEATURES=X_subset.columns.tolist()

In [ ]:
scaler_path = "E:/data/geo_k_compressed_raw_counts_enh/scaler_encoder.pkl"

if __name__ == "__main__":
    classifier = TabNetBinaryClassifier(
        X_original=X_subset,
        y_original=y_subset,
        scaler_path=None,
        n_d=77,
        n_a=110,
        n_steps=8,
        optimizer_fn=torch.optim.Adam,
        mask_type='entmax',
        n_independent=1,
        n_shared=5,
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        gamma=1.9,
        #clip_value=1,
        lambda_sparse=0.0005807365274596347,
        optimizer_params=dict(lr=0.003944458659837453),
        scheduler_params=dict(step_size=31, gamma=0.953213126517786),
        
        device_name='auto'  # Automatically detect GPU
    )
    
    # Show GPU info
    classifier.get_gpu_memory_info()
    
    # Train model 
    model = classifier.train(
        max_epochs=200, 
        patience=20, 
        batch_size=512,  # Larger batch size for GPU
        virtual_batch_size=128,
        num_workers=-1  # Parallel data loading
    )


In [ ]:
# Evaluate performance
results = classifier.evaluate()
    
classifier.plot_feature_importance(max_features=20)

In [ ]:
# Show model summary
classifier.get_model_summary()

# Save model
classifier.save_model(f'tabnet_binary_{MODEL_NAME}_{DATE}.zip')

# Clear GPU memory
classifier.clear_gpu_memory()

### Hypperparameters search with optuna



In [30]:
!pip install optuna[full]

  Using cached optuna-4.5.0-py3-none-any.whl.metadata (17 kB)
  Using cached colorlog-6.9.0-py3-none-any.whl.metadata (10 kB)
Using cached optuna-4.5.0-py3-none-any.whl (400 kB)
Using cached colorlog-6.9.0-py3-none-any.whl (11 kB)

   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   ---------------------------------------- 2/2 [optuna]



In [31]:
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
import numpy as np
import torch
from sklearn.metrics import roc_auc_score, accuracy_score

In [ ]:
"""{'n_d': 128, 
'n_a': 128, 
'n_steps': 7,
'gamma': 1.7928028379047916,
'n_independent': 3, 
'n_shared': 2,
'lambda_sparse': 0.006173448324113825,
'lr': 0.003775157784702449,
'step_size': 48,
'scheduler_gamma': 0.9035812858076991,
'batch_size': 256,
'virtual_batch_size': 128}"""

In [32]:
FEATURES = X_subset.columns.tolist()


In [ ]:
"""# Best Accuracy 
{'n_d': 128, 
'n_a': 128, 
'n_steps': 7,
'gamma': 1.7928028379047916,
'n_independent': 3, 
'n_shared': 2,
'lambda_sparse': 0.006173448324113825,
'lr': 0.003775157784702449,
'step_size': 48,
'scheduler_gamma': 0.9035812858076991,
    'batch_size': 256,
    'virtual_batch_size': 128}"""

In [ ]:
"""=== OPTIMIZATION COMPLETED ===
Best auc: 0.9657
Best parameters:
  - n_d: 77
  - n_a: 110
  - n_steps: 8
  - gamma: 1.9067107319494947
  - n_independent: 1
  - n_shared: 5
  - lambda_sparse: 0.0005807365274596347
  - lr: 0.003944458659837453
  - step_size: 31
  - scheduler_gamma: 0.953213126517786
  - batch_size: 512
  - virtual_batch_size: 128
  """

In [33]:
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
import numpy as np
import torch
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, confusion_matrix, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from pytorch_tabnet.tab_model import TabNetClassifier
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

class TabNetBinaryClassifierOptuna:
    """
    Class for training a TabNet binary classifier with GPU support and Optuna hyperparameter optimization
    """
    
    def __init__(self,
                 X_original=None,
                 y_original=None,
                 scaler_path=None,
                 features=None,
                 n_d=32,
                 n_a=32,
                 n_steps=5,
                 gamma=1.3,
                 n_independent=2,
                 n_shared=2,
                 lambda_sparse=1e-3,
                 optimizer_fn=torch.optim.Adam,
                 optimizer_params=dict(lr=1e-2),
                 mask_type='entmax',
                 scheduler_params=dict(step_size=50, gamma=0.9),
                 scheduler_fn=torch.optim.lr_scheduler.StepLR,
                 epsilon=1e-15,
                 device_name='auto'):
        """
        Initialize the TabNet classifier
        
        Parameters:
        -----------
        X_original : DataFrame or array
            Original features
        y_original : Series or array
            Original target
        scaler_path : str
            Path to saved scaler (optional)
        features : list
            List of feature names to scale
        n_d : int
            Dimension of learned representations
        n_a : int 
            Dimension of attention
        n_steps : int
            Number of steps in feature selection
        gamma : float
            Coefficient for aggregated attention
        lambda_sparse : float
            Regularization coefficient for sparsity
        device_name : str
            'auto', 'cuda', 'cpu' or specific device ('cuda:0')
        """
        
        # Device configuration
        self.device = self._setup_device(device_name)
        print(f"Device used: {self.device}")
        
        self.tabnet_params = {
            'n_d': n_d,
            'n_a': n_a, 
            'n_steps': n_steps,
            'gamma': gamma,
            'n_independent': n_independent,
            'n_shared': n_shared,
            'lambda_sparse': lambda_sparse,
            'optimizer_fn': optimizer_fn,
            'optimizer_params': optimizer_params,
            'mask_type': mask_type,
            'scheduler_params': scheduler_params,
            'scheduler_fn': scheduler_fn,
            'epsilon': epsilon,
            'device_name': self.device
        }
        
        self.model = None
        self.feature_names = None
        self.is_fitted = False
        self.best_params = None
        self.study = None
        self.X_original = X_original
        self.y_original = y_original
        self.scaler_path = scaler_path
        self.features = features

    def _setup_device(self, device_name):
        """Configure the computing device (CPU/GPU)"""
        if device_name == 'auto':
            if torch.cuda.is_available():
                device = 'cuda'
                print(f"GPU available: {torch.cuda.get_device_name()}")
                print(f"GPU memory available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
            else:
                device = 'cpu'
                print("GPU not available, using CPU")
        else:
            device = device_name
            if device.startswith('cuda') and not torch.cuda.is_available():
                print("WARNING: GPU requested but not available, using CPU")
                device = 'cpu'
        
        return device
    
    def get_gpu_memory_info(self):
        """Returns GPU memory information"""
        if torch.cuda.is_available() and self.device.startswith('cuda'):
            device_idx = 0 if self.device == 'cuda' else int(self.device.split(':')[1])
            allocated = torch.cuda.memory_allocated(device_idx) / 1e9
            reserved = torch.cuda.memory_reserved(device_idx) / 1e9
            total = torch.cuda.get_device_properties(device_idx).total_memory / 1e9
            
            print(f"GPU Memory:")
            print(f"  - Allocated: {allocated:.2f} GB")
            print(f"  - Reserved: {reserved:.2f} GB") 
            print(f"  - Total: {total:.2f} GB")
            print(f"  - Free: {total - reserved:.2f} GB")
            
            return {
                'allocated': allocated,
                'reserved': reserved,
                'total': total,
                'free': total - reserved
            }
        else:
            print("GPU memory not available")
            return None
    
    def clear_gpu_memory(self):
        """Clear GPU memory"""
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    def prepare_data(self, X=None, y=None, test_size=0.2, random_state=42):
        """Prepare data for training"""
        
        X = self.X_original if X is None else X
        y = self.y_original if y is None else y

        # Save feature names
        if hasattr(X, 'columns'):
            self.feature_names = X.columns.tolist()
        
        # Convert to float32 to optimize GPU memory
        if isinstance(X, pd.DataFrame):
            X = X.astype(np.float32)
        
        # Split into train (64%), validation (16%), and test (20%) sets
        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=random_state
        )

        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp, test_size=0.2, stratify=y_temp, random_state=random_state
        )

        # Reset indices
        if isinstance(X_train, pd.DataFrame):
            X_train = X_train.reset_index(drop=True)
            X_val = X_val.reset_index(drop=True)
            X_test = X_test.reset_index(drop=True)
        if isinstance(y_train, pd.Series):
            y_train = y_train.reset_index(drop=True)
            y_val = y_val.reset_index(drop=True)
            y_test = y_test.reset_index(drop=True)

        # Apply scaler if provided
        if self.scaler_path is not None and self.features is not None:
            print(f"Loading scaler from: {self.scaler_path}")
            scaler = joblib.load(self.scaler_path)
            
            column_trans = ColumnTransformer(
                [('scaler', scaler, self.features)], 
                remainder='passthrough', 
                n_jobs=-1
            )

            X_train = column_trans.fit_transform(X_train, y_train)
            X_val = column_trans.transform(X_val)
            X_test = column_trans.transform(X_test)

        # Convert to numpy arrays with appropriate dtypes
        self.X_train = X_train.values.astype(np.float32) if hasattr(X_train, 'values') else X_train.astype(np.float32)
        self.X_val = X_val.values.astype(np.float32) if hasattr(X_val, 'values') else X_val.astype(np.float32)
        self.X_test = X_test.values.astype(np.float32) if hasattr(X_test, 'values') else X_test.astype(np.float32)
        
        self.y_train = y_train.values.astype(np.int32) if hasattr(y_train, 'values') else y_train.astype(np.int32)
        self.y_val = y_val.values.astype(np.int32) if hasattr(y_val, 'values') else y_val.astype(np.int32)
        self.y_test = y_test.values.astype(np.int32) if hasattr(y_test, 'values') else y_test.astype(np.int32)

        print("Data prepared:")
        print(f"  - Training set: {self.X_train.shape}")
        print(f"  - Validation set: {self.X_val.shape}")
        print(f"  - Test set: {self.X_test.shape}")
        
        return self.X_train, self.X_val, self.X_test, self.y_train, self.y_val, self.y_test

    def optimize_hyperparameters(self, 
                                n_trials=50,
                                study_name=None,
                                metric='auc',
                                direction='maximize',
                                pruning=True,
                                timeout=None,
                                max_epochs_optuna=50,
                                patience_optuna=10):
        """
        Optimize hyperparameters using Optuna with enhanced parameter ranges
        
        Parameters:
        -----------
        n_trials : int
            Number of optimization trials
        study_name : str
            Name for the study (optional)
        metric : str
            Metric to optimize ('auc' or 'accuracy')
        direction : str
            'maximize' or 'minimize'
        pruning : bool
            Whether to use pruning for early trial termination
        timeout : int
            Time limit in seconds (None for no limit)
        max_epochs_optuna : int
            Max epochs for each trial
        patience_optuna : int
            Patience for each trial
        """

        # Prepare data
        self.prepare_data()

        print("\n=== STARTING HYPERPARAMETER OPTIMIZATION ===")
        print(f"Trials: {n_trials}")
        print(f"Metric: {metric}")
        print(f"Direction: {direction}")
        print(f"Max epochs per trial: {max_epochs_optuna}")
        
        def objective(trial):
            """Objective function for Optuna optimization"""
            
            # PARAMETRI ARCHITETTURALI MIGLIORATI
            n_d = trial.suggest_categorical('n_d', [64, 128, 256])
            n_a = trial.suggest_categorical('n_a', [64, 128, 256])
            n_steps = trial.suggest_int('n_steps', 3, 10)
            gamma = trial.suggest_float('gamma', 1.0, 2.0, step=0.1)
            
            # Assicura n_shared >= n_independent
            n_independent = trial.suggest_int('n_independent', 1, 3)
            n_shared = trial.suggest_int('n_shared', n_independent, 6)
            
            # REGOLARIZZAZIONE MIGLIORATA
            lambda_sparse = trial.suggest_float('lambda_sparse', 1e-6, 1e-3, log=True)
            momentum = trial.suggest_float('momentum', 0.02, 0.4, step=0.04)
            clip_value = trial.suggest_float('clip_value', 0.5, 2.0, step=0.5)
            
            # LEARNING RATE E OPTIMIZER
            lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
            
            # SCHEDULER - Scelta tra StepLR e ReduceLROnPlateau
            scheduler_type = trial.suggest_categorical('scheduler_type', ['step', 'plateau'])
            
            if scheduler_type == 'step':
                scheduler_params = {
                    'step_size': trial.suggest_int('step_size', 10, 50, step=5),
                    'gamma': trial.suggest_float('scheduler_gamma', 0.7, 0.95, step=0.02)
                }
                scheduler_fn = torch.optim.lr_scheduler.StepLR
            else:
                scheduler_params = {
                    'mode': 'min',
                    'factor': trial.suggest_float('scheduler_factor', 0.5, 0.9),
                    'patience': trial.suggest_int('scheduler_patience', 5, 15),
                    'min_lr': 1e-6
                }
                scheduler_fn = torch.optim.lr_scheduler.ReduceLROnPlateau
            
            # MASK TYPE
            mask_type = trial.suggest_categorical('mask_type', ['entmax', 'sparsemax'])
            
            # BATCH PARAMETERS
            batch_size = trial.suggest_categorical('batch_size', [256, 512, 1024, 2048])
            virtual_batch_size = trial.suggest_categorical('virtual_batch_size', [128, 256, 512])
            
            # Assicura virtual_batch_size <= batch_size
            if virtual_batch_size > batch_size:
                virtual_batch_size = batch_size
            
            # Costruisci parametri
            params = {
                'n_d': n_d,
                'n_a': n_a,
                'n_steps': n_steps,
                'gamma': gamma,
                'n_independent': n_independent,
                'n_shared': n_shared,
                'lambda_sparse': lambda_sparse,
                'momentum': momentum,
                'clip_value': clip_value,
                'optimizer_fn': torch.optim.AdamW,
                'optimizer_params': {'lr': lr},
                'mask_type': mask_type,
                'scheduler_params': scheduler_params,
                'scheduler_fn': scheduler_fn,
                'epsilon': 1e-15,
                'device_name': self.device,
                'verbose': 0
            }
            
            # Create temporary model
            temp_model = TabNetClassifier(**params)
            
            try:
                # Clear GPU memory before each trial
                if self.device.startswith('cuda'):
                    self.clear_gpu_memory()
                
                # Train model
                temp_model.fit(
                    X_train=self.X_train,
                    y_train=self.y_train.reshape(-1),
                    eval_set=[(self.X_val, self.y_val.reshape(-1))],
                    eval_name=['val'],
                    eval_metric=['auc'],#['accuracy', 'auc'],
                    max_epochs=max_epochs_optuna,
                    patience=patience_optuna,
                    batch_size=batch_size,
                    virtual_batch_size=virtual_batch_size,
                    num_workers=0,
                    drop_last=False
                )
                
                # Make predictions on validation set
                y_pred_proba = temp_model.predict_proba(self.X_val)
                y_pred = temp_model.predict(self.X_val)
                
                # Calculate metrics
                if metric == 'auc':
                    score = roc_auc_score(self.y_val, y_pred_proba[:, 1])
                elif metric == 'accuracy':
                    score = accuracy_score(self.y_val, y_pred)
                else:
                    raise ValueError(f"Unsupported metric: {metric}")
                
                # Report intermediate values for pruning
                trial.report(score, step=max_epochs_optuna)
                
                # Handle pruning
                if trial.should_prune():
                    raise optuna.exceptions.TrialPruned()
                
                return score
                
            except Exception as e:
                print(f"Trial failed: {str(e)}")
                return 0.0 if direction == 'maximize' else float('inf')
            
            finally:
                # Clean up memory
                del temp_model
                if self.device.startswith('cuda'):
                    self.clear_gpu_memory()
        
        # Create study with MedianPruner
        sampler = TPESampler(seed=42)
        pruner = MedianPruner(
            n_startup_trials=10,
            n_warmup_steps=20,
            interval_steps=10
        ) if pruning else None
        
        study_name = study_name or f"tabnet_optimization_{metric}"
        self.study = optuna.create_study(
            direction=direction,
            sampler=sampler,
            pruner=pruner,
            study_name=study_name
        )
        
        # Run optimization
        print("\nRunning optimization...")
        self.study.optimize(
            objective, 
            n_trials=n_trials,
            n_jobs=1,  # TabNet is not thread-safe
            timeout=timeout,
            show_progress_bar=True
        )
        
        # Store best parameters
        self.best_params = self.study.best_params.copy()
        
        # Print results
        print(f"\n{'='*60}")
        print("OPTIMIZATION COMPLETED")
        print(f"{'='*60}")
        print(f"Best {metric}: {self.study.best_value:.4f}")
        print(f"\nBest parameters:")
        for key, value in self.best_params.items():
            print(f"  {key}: {value}")
        
        print("\nOptimization statistics:")
        print(f"  - Total trials: {len(self.study.trials)}")
        print(f"  - Completed: {len([t for t in self.study.trials if t.state == optuna.trial.TrialState.COMPLETE])}")
        print(f"  - Pruned: {len([t for t in self.study.trials if t.state == optuna.trial.TrialState.PRUNED])}")
        print(f"  - Failed: {len([t for t in self.study.trials if t.state == optuna.trial.TrialState.FAIL])}")
        print(f"{'='*60}")
        
        return self.study
    
    def train_with_best_params(self, 
                              max_epochs=200, 
                              patience=20,
                              num_workers=0,
                              drop_last=False):
        """Train model with best parameters found by Optuna"""
        if self.best_params is None:
            raise ValueError("You must first run optimize_hyperparameters()")
        
        print("\n=== TRAINING WITH BEST PARAMETERS ===")
        
        # Extract training parameters
        batch_size = self.best_params.pop('batch_size', 1024)
        virtual_batch_size = self.best_params.pop('virtual_batch_size', 128)
        lr = self.best_params.pop('lr', 1e-2)
        scheduler_type = self.best_params.pop('scheduler_type', 'step')
        
        # Build scheduler params based on type
        if scheduler_type == 'step':
            step_size = self.best_params.pop('step_size', 50)
            scheduler_gamma = self.best_params.pop('scheduler_gamma', 0.9)
            scheduler_params = {'step_size': step_size, 'gamma': scheduler_gamma}
            scheduler_fn = torch.optim.lr_scheduler.StepLR
            # Remove plateau-specific params if present
            self.best_params.pop('scheduler_factor', None)
            self.best_params.pop('scheduler_patience', None)
        else:
            scheduler_factor = self.best_params.pop('scheduler_factor', 0.7)
            scheduler_patience = self.best_params.pop('scheduler_patience', 10)
            scheduler_params = {
                'mode': 'min',
                'factor': scheduler_factor,
                'patience': scheduler_patience,
                'min_lr': 1e-6
            }
            scheduler_fn = torch.optim.lr_scheduler.ReduceLROnPlateau
            # Remove step-specific params if present
            self.best_params.pop('step_size', None)
            self.best_params.pop('scheduler_gamma', None)
        
        # Update tabnet_params with best parameters
        self.tabnet_params.update(self.best_params)
        self.tabnet_params['optimizer_params'] = {'lr': lr}
        self.tabnet_params['scheduler_params'] = scheduler_params
        self.tabnet_params['scheduler_fn'] = scheduler_fn
        
        # Train with original method using best parameters
        return self.train(
            max_epochs=max_epochs,
            patience=patience,
            batch_size=batch_size,
            virtual_batch_size=virtual_batch_size,
            num_workers=num_workers,
            drop_last=drop_last
        )
    
    def train(self, 
              max_epochs=200, 
              patience=20, 
              batch_size=1024,
              virtual_batch_size=128,
              num_workers=0,
              drop_last=False):
        """Train the TabNet model"""
        
        # Prepare data if not already done
        if not hasattr(self, 'X_train'):
            self.prepare_data()
        
        # Adapt batch_size for GPU
        if self.device.startswith('cuda'):
            gpu_memory = self.get_gpu_memory_info()
            if gpu_memory and gpu_memory['free'] < 2.0:
                suggested_batch_size = min(batch_size, 512)
                print(f"Limited GPU memory, reducing batch_size to {suggested_batch_size}")
                batch_size = suggested_batch_size
        
        print("Training configuration:")
        print(f"  - Device: {self.device}")
        print(f"  - Batch size: {batch_size}")
        print(f"  - Virtual batch size: {virtual_batch_size}")
        print(f"  - Max epochs: {max_epochs}")
        print(f"  - Patience: {patience}")
        
        # Initialize model
        self.model = TabNetClassifier(**self.tabnet_params)
        
        # Clear GPU memory before training
        if self.device.startswith('cuda'):
            self.clear_gpu_memory()
            print("\nGPU memory before training:")
            self.get_gpu_memory_info()
        
        print("\nStarting TabNet training...") 
        
        try:
            self.model.fit(
                X_train=self.X_train,
                y_train=self.y_train.reshape(-1),
                eval_set=[(self.X_val, self.y_val.reshape(-1))],
                eval_name=['val'],
                eval_metric=['accuracy', 'auc'],
                max_epochs=max_epochs,
                patience=patience,
                batch_size=batch_size,
                virtual_batch_size=virtual_batch_size,
                num_workers=num_workers,
                drop_last=drop_last
            )
            
            self.is_fitted = True
            print("Training completed!")
            
            # Check memory after training
            if self.device.startswith('cuda'):
                print("\nGPU memory after training:")
                self.get_gpu_memory_info()
                
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print("\nERROR: Insufficient GPU memory!")
                self.clear_gpu_memory()
            raise e
        
        return self.model
    
    def predict(self, X=None):
        """Make predictions"""
        if not self.is_fitted:
            raise ValueError("Model has not been trained yet!")
        
        if X is None:
            X = self.X_test
            
        # Convert to float32
        if isinstance(X, pd.DataFrame):
            X = X.values.astype(np.float32)
        elif not isinstance(X, np.ndarray):
            X = np.array(X, dtype=np.float32)
        else:
            X = X.astype(np.float32)
            
        predictions = self.model.predict(X)
        return predictions
    
    def predict_proba(self, X=None):
        """Return prediction probabilities"""
        if not self.is_fitted:
            raise ValueError("Model has not been trained yet!")
        
        if X is None:
            X = self.X_test
            
        # Convert to float32
        if isinstance(X, pd.DataFrame):
            X = X.values.astype(np.float32)
        elif not isinstance(X, np.ndarray):
            X = np.array(X, dtype=np.float32)
        else:
            X = X.astype(np.float32)
            
        probabilities = self.model.predict_proba(X)
        return probabilities
    
    def evaluate(self, X=None, y=None, plot_results=True):
        """Evaluate model performance"""
        if not self.is_fitted:
            raise ValueError("Model has not been trained yet!")
        
        if X is None:
            X = self.X_test
            y = self.y_test
        
        # Predictions
        y_pred = self.predict(X)
        y_pred_proba = self.predict_proba(X)
        
        # Metrics
        accuracy = accuracy_score(y, y_pred)
        auc_score = roc_auc_score(y, y_pred_proba[:, 1])

        print(f"\n=== EVALUATION RESULTS ===")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"AUC Score: {auc_score:.4f}")
        print(f"\nClassification Report:")
        print(classification_report(y, y_pred))
        
        if plot_results:
            self.plot_results(y, y_pred, y_pred_proba[:, 1])
        
        return {
            'accuracy': accuracy,
            'auc_score': auc_score,
            'predictions': y_pred,
            'probabilities': y_pred_proba
        }
    
    def plot_results(self, y_true, y_pred, y_pred_proba):
        """Visualize results"""
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        # Confusion Matrix
        cm = confusion_matrix(y_true, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
        axes[0].set_title('Confusion Matrix')
        axes[0].set_xlabel('Predicted')
        axes[0].set_ylabel('Actual')
        
        # ROC Curve
        fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
        auc = roc_auc_score(y_true, y_pred_proba)
        
        axes[1].plot(fpr, tpr, label=f'ROC Curve (AUC = {auc:.3f})')
        axes[1].plot([0, 1], [0, 1], 'k--', label='Random')
        axes[1].set_xlabel('False Positive Rate')
        axes[1].set_ylabel('True Positive Rate')
        axes[1].set_title('ROC Curve')
        axes[1].legend()
        axes[1].grid(True)
        
        # Distribution of Probabilities
        axes[2].hist(y_pred_proba[y_true == 0], bins=30, alpha=0.7, label='Class 0', color='red')
        axes[2].hist(y_pred_proba[y_true == 1], bins=30, alpha=0.7, label='Class 1', color='blue')
        axes[2].set_xlabel('Predicted Probability')
        axes[2].set_ylabel('Frequency')
        axes[2].set_title('Distribution of Predicted Probabilities')
        axes[2].legend()
        axes[2].grid(True)
        
        plt.tight_layout()
        plt.show()
    
    def plot_optimization_history(self):
        """Plot optimization history"""
        if self.study is None:
            raise ValueError("No optimization study found. Run optimize_hyperparameters() first.")
        
        try:
            fig, axes = plt.subplots(2, 2, figsize=(15, 10))
            
            # Optimization history
            trials = self.study.trials
            values = [t.value for t in trials if t.value is not None]
            
            axes[0, 0].plot(values)
            axes[0, 0].set_title('Optimization History')
            axes[0, 0].set_xlabel('Trial')
            axes[0, 0].set_ylabel('Objective Value')
            axes[0, 0].grid(True)
            
            # Parameter importance
            try:
                importance = optuna.importance.get_param_importances(self.study)
                params = list(importance.keys())[:10]
                importances = [importance[p] for p in params]
                
                axes[0, 1].barh(params, importances)
                axes[0, 1].set_title('Parameter Importance (Top 10)')
                axes[0, 1].set_xlabel('Importance')
            except:
                axes[0, 1].text(0.5, 0.5, 'Parameter importance\nnot available', 
                               ha='center', va='center', transform=axes[0, 1].transAxes)
            
            # Correlation heatmap
            if len(trials) > 1:
                param_names = ['n_d', 'n_a', 'n_steps', 'lr', 'batch_size']
                trial_data = []
                for trial in trials:
                    if trial.value is not None:
                        row = [trial.value]
                        for param in param_names:
                            if param in trial.params:
                                row.append(trial.params[param])
                            else:
                                row.append(None)
                        trial_data.append(row)
                
                if trial_data:
                    df = pd.DataFrame(trial_data, columns=['objective'] + param_names)
                    df = df.dropna()
                    
                    if len(df) > 0:
                        corr = df.corr()
                        sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, ax=axes[1, 0])
                        axes[1, 0].set_title('Parameter Correlation')
            
            # Best trial info
            best_trial = self.study.best_trial
            axes[1, 1].text(0.1, 0.9, f'Best Trial: #{best_trial.number}', 
                           fontsize=12, fontweight='bold', transform=axes[1, 1].transAxes)
            axes[1, 1].text(0.1, 0.8, f'Best Value: {best_trial.value:.4f}', 
                           fontsize=11, transform=axes[1, 1].transAxes)
            
            y_pos = 0.7
            axes[1, 1].text(0.1, y_pos, 'Best Parameters:', 
                           fontsize=11, fontweight='bold', transform=axes[1, 1].transAxes)
            y_pos -= 0.08
            
            for key, value in list(best_trial.params.items())[:8]:
                axes[1, 1].text(0.1, y_pos, f'{key}: {value}', fontsize=9,
                               transform=axes[1, 1].transAxes)
                y_pos -= 0.06
            
            axes[1, 1].set_xlim(0, 1)
            axes[1, 1].set_ylim(0, 1)
            axes[1, 1].axis('off')
            
            plt.tight_layout()
            plt.show()
            
        except ImportError:
            print("Matplotlib/Seaborn not available for plotting")
    
    def get_optimization_summary(self):
        """Get summary of optimization results"""
        if self.study is None:
            raise ValueError("No optimization study found. Run optimize_hyperparameters() first.")
        
        summary = {
            'best_value': self.study.best_value,
            'best_params': self.study.best_params,
            'n_trials': len(self.study.trials),
            'completed_trials': len([t for t in self.study.trials if t.state == optuna.trial.TrialState.COMPLETE]),
            'pruned_trials': len([t for t in self.study.trials if t.state == optuna.trial.TrialState.PRUNED]),
            'failed_trials': len([t for t in self.study.trials if t.state == optuna.trial.TrialState.FAIL]),
            'study_name': self.study.study_name
        }
        
        return summary
    
    def save_model(self, filepath):
        """Save the model"""
        if not self.is_fitted:
            raise ValueError("Model has not been trained yet!")
        
        self.model.save_model(filepath)
        print(f"Model saved at: {filepath}")
    
    def load_model(self, filepath):
        """Load a saved model"""
        self.model = TabNetClassifier(device_name=self.device)
        self.model.load_model(filepath)
        self.is_fitted = True
        print(f"Model loaded from: {filepath}")
    
    def get_model_summary(self):
        """Return model and hardware summary"""
        if not self.is_fitted:
            print("Model not yet trained")
            return
        
        print("\n=== MODEL SUMMARY ===")
        print(f"Device: {self.device}")
        print("\nTabNet Parameters:")
        for key, value in self.tabnet_params.items():
            if key != 'device_name':
                print(f"  - {key}: {value}")
        
        if self.best_params:
            print("\nOptimized Parameters:")
            for key, value in self.best_params.items():
                print(f"  - {key}: {value}")
        
        if hasattr(self.model, 'network'):
            total_params = sum(p.numel() for p in self.model.network.parameters())
            trainable_params = sum(p.numel() for p in self.model.network.parameters() if p.requires_grad)
            print("\nModel Parameters:")
            print(f"  - Total: {total_params:,}")
            print(f"  - Trainable: {trainable_params:,}")
        
        if self.device.startswith('cuda'):
            print()
            self.get_gpu_memory_info()



In [37]:
# Create a stratified subset for optuna optimization
X_subset_optuna, _, y_subset_optuna, _ = train_test_split(
    X_subset,
    y_subset,
    test_size=0.7,
    stratify=y_subset,
    random_state=42
)
FEATURES = X_subset.columns.tolist()

# Reset index for convenience
X_subset_optuna = X_subset_optuna.reset_index(drop=True)
y_subset_optuna = y_subset_optuna.reset_index(drop=True)

In [38]:
X_subset_optuna.shape, y_subset_optuna.shape

((1425000, 20), (1425000, 1))

In [39]:
# Initialize the classifier
classifier = TabNetBinaryClassifierOptuna(
    X_original=X_subset_optuna,
    y_original=y_subset_optuna['0'],
    scaler_path=None,  # Optional
    features=None,   # Features to scale
    device_name='auto'
)

# Run hyperparameter optimization
study = classifier.optimize_hyperparameters(
    n_trials=100,
    metric='auc',
    direction='maximize',
    max_epochs_optuna=20,
    patience_optuna=10,
    #timeout=3600  # 1 hour
)

GPU available: NVIDIA GeForce RTX 5090
GPU memory available: 34.2 GB
Device used: cuda


[I 2025-09-26 16:44:23,328] A new study created in memory with name: tabnet_optimization_auc


Data prepared:
  - Training set: (912000, 20)
  - Validation set: (228000, 20)
  - Test set: (285000, 20)

=== STARTING HYPERPARAMETER OPTIMIZATION ===
Trials: 100
Metric: auc
Direction: maximize
Max epochs per trial: 20

Running optimization...


  0%|          | 0/100 [00:00<?, ?it/s]

Stop training because you reached max_epochs = 20 with best_epoch = 19 and best_val_auc = 0.90706


...

Best trial: 2. Best value: 0.945019:   4%|▍         | 4/100 [1:12:18<24:29:11, 918.24s/it] 

[I 2025-09-26 17:56:42,062] Trial 3 finished with value: 0.9088430193521083 and parameters: {'n_d': 64, 'n_a': 256, 'n_steps': 8, 'gamma': 1.9, 'n_independent': 2, 'n_shared': 2, 'lambda_sparse': 0.00013795402040204168, 'momentum': 0.30000000000000004, 'clip_value': 1.5, 'lr': 0.002055424552015075, 'scheduler_type': 'plateau', 'scheduler_factor': 0.6710164073434198, 'scheduler_patience': 5, 'mask_type': 'entmax', 'batch_size': 2048, 'virtual_batch_size': 512}. Best is trial 2 with value: 0.945018563173284.


Best trial: 2. Best value: 0.945019:   4%|▍         | 4/100 [1:22:06<32:50:28, 1231.54s/it]

[W 2025-09-26 18:06:29,502] Trial 4 failed with parameters: {'n_d': 256, 'n_a': 128, 'n_steps': 8, 'gamma': 1.9, 'n_independent': 3, 'n_shared': 3, 'lambda_sparse': 0.00047607677518095016, 'momentum': 0.22, 'clip_value': 2.0, 'lr': 0.004878360603452143, 'scheduler_type': 'step', 'step_size': 20, 'scheduler_gamma': 0.7999999999999999, 'mask_type': 'sparsemax', 'batch_size': 512, 'virtual_batch_size': 512} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\atogni\anaconda3\envs\geok_5090\Lib\site-packages\optuna\study\_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\atogni\AppData\Local\Temp\ipykernel_17792\1946839435.py", line 330, in objective
    temp_model.fit(
  File "c:\Users\atogni\anaconda3\envs\geok_5090\Lib\site-packages\pytorch_tabnet\abstract_model.py", line 258, in fit
    self._train_epoch(train_dataloader)
  File "c:\Users\atogni\anaconda3\envs\g

KeyboardInterrupt: 

In [ ]:
# Visualize optimization results
classifier.plot_optimization_history()

# Get optimization summary
summary = classifier.get_optimization_summary()
print(summary)

In [ ]:
# Train final model with best parameters
classifier.train_with_best_params(
    max_epochs=200,
    patience=20
)

# Evaluate on test set
results = classifier.evaluate(plot_results=True)

# Get model summary
classifier.get_model_summary()

# Save model
classifier.save_model('best_tabnet_model.zip')

# Optional: Save study results
study.trials_dataframe().to_csv('optimization_results.csv', index=False)

## Deep test

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report, matthews_corrcoef,
    average_precision_score, log_loss, brier_score_loss
)
from sklearn.model_selection import StratifiedKFold
import warnings
warnings.filterwarnings('ignore')

class DeepTest:
    """
    Classe avanzata per testare modelli TabNet su múltipli test set
    con metriche complete e visualizzazioni professionali
    """
    
    def __init__(self, model_path, test_data, test_labels):
        """
        Inizializza DeepTest
        
        Parameters:
        -----------
        model_path : str
            Path del modello TabNet salvato (.zip)
        test_data : pd.DataFrame
            DataFrame con i dati di test
        test_labels : np.array
            Array con le label di test
        """
        self.model_path = model_path
        self.test_data = test_data
        self.test_labels = test_labels
        self.model = None
        self.results = {}
        self.detailed_results = []
        
        # Configurazione colori per grafici professionali
        self.colors = {
            'primary': '#2E86AB',
            'secondary': '#A23B72', 
            'accent': '#F18F01',
            'success': '#C73E1D',
            'neutral': '#6C757D',
            'light': '#F8F9FA'
        }
        
        # Stile professionale
        plt.style.use('default')
        sns.set_palette([self.colors['primary'], self.colors['secondary'], 
                        self.colors['accent'], self.colors['success']])
        
        print(" DeepTest inizializzato")
        print(f"   📊 Dati test: {test_data.shape}")
        print(f"   🎯 Label: {len(test_labels)} ({np.sum(test_labels)} positivi)")
    
    def load_model(self, tabnet_classifier_class):
        """
        Carica il modello TabNet
        
        Parameters:
        -----------
        tabnet_classifier_class : class
            Classe TabNetBinaryClassifier
        """
        try:
            # Crea un'istanza dummy per il caricamento
            self.model = tabnet_classifier_class(
                X_original=self.test_data.iloc[:10],  # dummy data
                y_original=pd.Series([0]*10),  # dummy labels
                scaler_path=None,
                n_d=32, n_a=32, n_steps=3, gamma=1.3
            )
            
            # Carica il modello salvato
            self.model.load_model(self.model_path)
            print(f"✅ Modello caricato da: {self.model_path}")
            
        except Exception as e:
            print(f"Errore nel caricamento del modello: {e}")
            raise
    
    def create_test_splits(self, n_splits=10, random_state=42):
        """
        Crea N split stratificati dei dati di test
        
        Parameters:
        -----------
        n_splits : int
            Numero di split da creare
        random_state : int
            Seed per riproducibilità
            
        Returns:
        --------
        list : Lista di tuple (X_split, y_split)
        """
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        splits = []
        
        for train_idx, test_idx in skf.split(self.test_data, self.test_labels):
            X_split = self.test_data.iloc[test_idx].copy()
            y_split = self.test_labels[test_idx].copy()
            splits.append((X_split, y_split))
        
        print(f"📋 Creati {n_splits} split stratificati")
        return splits
    
    def calculate_metrics(self, y_true, y_pred, y_pred_proba):
        """
        Calcola tutte le metriche di classificazione
        
        Returns:
        --------
        dict : Dizionario con tutte le metriche
        """
        metrics = {}
        
        # Metriche base
        metrics['accuracy'] = accuracy_score(y_true, y_pred)
        metrics['precision'] = precision_score(y_true, y_pred, zero_division=0)
        metrics['recall'] = recall_score(y_true, y_pred, zero_division=0)
        metrics['f1'] = f1_score(y_true, y_pred, zero_division=0)
        metrics['specificity'] = precision_score(y_true, y_pred, pos_label=0, zero_division=0)
        
        # Metriche avanzate
        try:
            metrics['auc_roc'] = roc_auc_score(y_true, y_pred_proba)
            metrics['auc_pr'] = average_precision_score(y_true, y_pred_proba)
            metrics['log_loss'] = log_loss(y_true, y_pred_proba)
            metrics['brier_score'] = brier_score_loss(y_true, y_pred_proba)
        except:
            metrics['auc_roc'] = 0.5
            metrics['auc_pr'] = np.mean(y_true)
            metrics['log_loss'] = np.inf
            metrics['brier_score'] = np.inf
        
        # Matthews Correlation Coefficient
        metrics['mcc'] = matthews_corrcoef(y_true, y_pred)
        
        # Matrice di confusione
        cm = confusion_matrix(y_true, y_pred)
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            metrics['true_positives'] = tp
            metrics['true_negatives'] = tn
            metrics['false_positives'] = fp
            metrics['false_negatives'] = fn
            metrics['sensitivity'] = tp / (tp + fn) if (tp + fn) > 0 else 0
            metrics['specificity'] = tn / (tn + fp) if (tn + fp) > 0 else 0
        else:
            metrics.update({
                'true_positives': 0, 'true_negatives': 0,
                'false_positives': 0, 'false_negatives': 0,
                'sensitivity': 0, 'specificity': 0
            })
        
        return metrics
    
    def run_comprehensive_test(self, n_splits=10, verbose=True):
        """
        Esegue test completo su N split con tutte le metriche
        
        Parameters:
        -----------
        n_splits : int
            Numero di split per il test
        verbose : bool
            Mostra progress durante il test
            
        Returns:
        --------
        dict : Risultati aggregati del test
        """
        if self.model is None:
            raise ValueError("❌ Carica prima il modello con load_model()")
        
        print(f"\n🔬 Inizio test completo su {n_splits} split...")
        
        # Crea gli split
        splits = self.create_test_splits(n_splits)
        
        # Lista per salvare tutti i risultati
        all_metrics = []
        all_predictions = []
        all_probabilities = []
        all_true_labels = []
        
        # Testa su ogni split
        for i, (X_split, y_split) in enumerate(splits):
            if verbose:
                print(f"   Split {i+1}/{n_splits}: {X_split.shape[0]} samples", end=" -> ")
            
            try:
                # Predizioni
                y_pred = self.model.predict(X_split)
                y_pred_proba = self.model.predict_proba(X_split)[:, 1]
                
                # Calcola metriche
                metrics = self.calculate_metrics(y_split, y_pred, y_pred_proba)
                metrics['split_id'] = i
                metrics['n_samples'] = len(y_split)
                
                all_metrics.append(metrics)
                all_predictions.extend(y_pred)
                all_probabilities.extend(y_pred_proba)
                all_true_labels.extend(y_split)
                
                if verbose:
                    print(f"Acc: {metrics['accuracy']:.3f}, AUC: {metrics['auc_roc']:.3f}")
                
            except Exception as e:
                print(f"❌ Errore nello split {i+1}: {e}")
                continue
        
        # Converti in DataFrame per analisi più facile
        metrics_df = pd.DataFrame(all_metrics)
        
        # Calcola statistiche aggregate
        aggregate_stats = {}
        metric_names = [col for col in metrics_df.columns 
                       if col not in ['split_id', 'n_samples']]
        
        for metric in metric_names:
            aggregate_stats[f'{metric}_mean'] = metrics_df[metric].mean()
            aggregate_stats[f'{metric}_std'] = metrics_df[metric].std()
            aggregate_stats[f'{metric}_min'] = metrics_df[metric].min()
            aggregate_stats[f'{metric}_max'] = metrics_df[metric].max()
        
        # Salva risultati
        self.results = {
            'individual_results': metrics_df,
            'aggregate_stats': aggregate_stats,
            'all_predictions': np.array(all_predictions),
            'all_probabilities': np.array(all_probabilities),
            'all_true_labels': np.array(all_true_labels),
            'n_splits': n_splits
        }
        
        print("Test completato!")
        print(f"Accuracy media: {aggregate_stats['accuracy_mean']:.4f} ± {aggregate_stats['accuracy_std']:.4f}")
        print(f"AUC media: {aggregate_stats['auc_roc_mean']:.4f} ± {aggregate_stats['auc_roc_std']:.4f}")
        
        return self.results
    
    def print_detailed_report(self):
        """
        Stampa report dettagliato dei risultati
        """
        if not self.results:
            print("❌ Esegui prima run_comprehensive_test()")
            return
        
        stats = self.results['aggregate_stats']
        
        print("\n" + "="*60)
        print("DEEP TEST REPORT - METRICHE AGGREGATE")
        print("="*60)
        
        # Metriche principali
        main_metrics = [
            ('Accuracy', 'accuracy'),
            ('Precision', 'precision'),
            ('Recall', 'recall'),
            ('F1-Score', 'f1'),
            ('AUC-ROC', 'auc_roc'),
            ('AUC-PR', 'auc_pr'),
            ('MCC', 'mcc')
        ]
        
        print("METRICHE PRINCIPALI:")
        for name, key in main_metrics:
            mean_val = stats[f'{key}_mean']
            std_val = stats[f'{key}_std']
            print(f"   {name:<12}: {mean_val:.4f} ± {std_val:.4f}")
        
        # Confusion Matrix aggregata
        print(f"CONFUSION MATRIX (Media su {self.results['n_splits']} split):")
        tp = stats['true_positives_mean']
        tn = stats['true_negatives_mean'] 
        fp = stats['false_positives_mean']
        fn = stats['false_negatives_mean']
        
        print(f"   True Positives : {tp:.1f} ± {stats['true_positives_std']:.1f}")
        print(f"   True Negatives : {tn:.1f} ± {stats['true_negatives_std']:.1f}")
        print(f"   False Positives: {fp:.1f} ± {stats['false_positives_std']:.1f}")
        print(f"   False Negatives: {fn:.1f} ± {stats['false_negatives_std']:.1f}")
        
        # Metriche di loss
        print(f"METRICHE DI LOSS:")
        print(f"   Log Loss    : {stats['log_loss_mean']:.4f} ± {stats['log_loss_std']:.4f}")
        print(f"   Brier Score : {stats['brier_score_mean']:.4f} ± {stats['brier_score_std']:.4f}")
    
    def create_professional_visualizations(self, figsize=(20, 15), save_path=None):
        """
        Crea visualizzazioni professionali per la presentazione
        
        Parameters:
        -----------
        figsize : tuple
            Dimensioni della figura
        save_path : str
            Path per salvare la figura (opzionale)
        """
        if not self.results:
            print("Esegui prima run_comprehensive_test()")
            return
        
        # Configura la figura
        fig = plt.figure(figsize=figsize)
        fig.suptitle('TabNet Model - Comprehensive Performance Analysis', 
                     fontsize=20, fontweight='bold', y=0.98)
        
        # Layout: 3x3 grid
        gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3, 
                             left=0.06, right=0.94, top=0.92, bottom=0.06)
        
        # 1. Metriche principali - Boxplot
        ax1 = fig.add_subplot(gs[0, :2])
        self._plot_metrics_boxplot(ax1)
        
        # 2. Confusion Matrix media
        ax2 = fig.add_subplot(gs[0, 2])
        self._plot_avg_confusion_matrix(ax2)
        
        # 3. ROC Curve
        ax3 = fig.add_subplot(gs[1, 0])
        self._plot_roc_curve(ax3)
        
        # 4. Precision-Recall Curve  
        ax4 = fig.add_subplot(gs[1, 1])
        self._plot_precision_recall_curve(ax4)
        
        # 5. Distribuzione delle probabilità
        ax5 = fig.add_subplot(gs[1, 2])
        self._plot_probability_distribution(ax5)
        
        # 6. Stabilità delle metriche
        ax6 = fig.add_subplot(gs[2, 0])
        self._plot_metrics_stability(ax6)
        
        # 7. Accuracy vs AUC scatter
        ax7 = fig.add_subplot(gs[2, 1])
        self._plot_accuracy_vs_auc(ax7)
        
        # 8. Tabella riassuntiva
        ax8 = fig.add_subplot(gs[2, 2])
        self._plot_summary_table(ax8)
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight', 
                       facecolor='white', edgecolor='none')
            print(f"Visualizzazioni salvate in: {save_path}")
        
        plt.tight_layout()
        plt.show()
    
    def _plot_metrics_boxplot(self, ax):
        """Boxplot delle metriche principali"""
        df = self.results['individual_results']
        metrics = ['accuracy', 'precision', 'recall', 'f1', 'auc_roc']
        
        data_to_plot = [df[metric].values for metric in metrics]
        labels = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC']
        
        bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True)
        
        colors = [self.colors['primary'], self.colors['secondary'], 
                 self.colors['accent'], self.colors['success'], self.colors['neutral']]
        
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        
        ax.set_title('Distribution of Key Metrics Across Test Splits', 
                    fontsize=14, fontweight='bold')
        ax.set_ylabel('Score')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1.05)
    
    def _plot_avg_confusion_matrix(self, ax):
        """Confusion matrix media"""
        stats = self.results['aggregate_stats']
        
        cm_data = np.array([
            [stats['true_negatives_mean'], stats['false_positives_mean']],
            [stats['false_negatives_mean'], stats['true_positives_mean']]
        ])
        
        sns.heatmap(cm_data, annot=True, fmt='.1f', cmap='Blues',
                   xticklabels=['Pred 0', 'Pred 1'],
                   yticklabels=['True 0', 'True 1'], ax=ax)
        ax.set_title('Average Confusion Matrix', fontsize=14, fontweight='bold')
    
    def _plot_roc_curve(self, ax):
        """ROC Curve aggregata"""
        y_true = self.results['all_true_labels']
        y_scores = self.results['all_probabilities']
        
        fpr, tpr, _ = roc_curve(y_true, y_scores)
        auc_score = roc_auc_score(y_true, y_scores)
        
        ax.plot(fpr, tpr, color=self.colors['primary'], linewidth=3,
               label=f'ROC Curve (AUC = {auc_score:.3f})')
        ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
        
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title('ROC Curve', fontsize=14, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    def _plot_precision_recall_curve(self, ax):
        """Precision-Recall Curve"""
        y_true = self.results['all_true_labels']
        y_scores = self.results['all_probabilities']
        
        precision, recall, _ = precision_recall_curve(y_true, y_scores)
        ap_score = average_precision_score(y_true, y_scores)
        
        ax.plot(recall, precision, color=self.colors['secondary'], linewidth=3,
               label=f'PR Curve (AP = {ap_score:.3f})')
        
        baseline = np.sum(y_true) / len(y_true)
        ax.axhline(y=baseline, color='k', linestyle='--', alpha=0.5,
                  label=f'Baseline = {baseline:.3f}')
        
        ax.set_xlabel('Recall')
        ax.set_ylabel('Precision')
        ax.set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    def _plot_probability_distribution(self, ax):
        """Distribuzione delle probabilità predette"""
        y_true = self.results['all_true_labels']
        y_scores = self.results['all_probabilities']
        
        ax.hist(y_scores[y_true == 0], bins=30, alpha=0.7, 
               color=self.colors['accent'], label='Class 0', density=True)
        ax.hist(y_scores[y_true == 1], bins=30, alpha=0.7,
               color=self.colors['primary'], label='Class 1', density=True)
        
        ax.set_xlabel('Predicted Probability')
        ax.set_ylabel('Density')
        ax.set_title('Probability Distribution by True Class', 
                    fontsize=14, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    def _plot_metrics_stability(self, ax):
        """Stabilità delle metriche across splits"""
        df = self.results['individual_results']
        
        metrics = ['accuracy', 'auc_roc', 'f1']
        colors = [self.colors['primary'], self.colors['secondary'], self.colors['accent']]
        
        for metric, color in zip(metrics, colors):
            ax.plot(df['split_id'], df[metric], 'o-', color=color, 
                   label=metric.upper(), linewidth=2, markersize=6)
        
        ax.set_xlabel('Test Split')
        ax.set_ylabel('Score')
        ax.set_title('Metrics Stability Across Splits', 
                    fontsize=14, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1.05)
    
    def _plot_accuracy_vs_auc(self, ax):
        """Scatter Accuracy vs AUC"""
        df = self.results['individual_results']
        
        scatter = ax.scatter(df['accuracy'], df['auc_roc'], 
                            c=df['f1'], cmap='viridis', 
                            s=100, alpha=0.7, edgecolors='black')
        
        ax.set_xlabel('Accuracy')
        ax.set_ylabel('AUC-ROC')
        ax.set_title('Accuracy vs AUC-ROC\n(colored by F1-score)', 
                    fontsize=14, fontweight='bold')
        
        # Linea di trend
        z = np.polyfit(df['accuracy'], df['auc_roc'], 1)
        p = np.poly1d(z)
        ax.plot(df['accuracy'], p(df['accuracy']), "r--", alpha=0.8)
        
        plt.colorbar(scatter, ax=ax, label='F1-Score')
        ax.grid(True, alpha=0.3)
    
    def _plot_summary_table(self, ax):
        """Tabella riassuntiva delle metriche"""
        stats = self.results['aggregate_stats']
        
        # Dati per la tabella
        table_data = [
            ['Accuracy', f"{stats['accuracy_mean']:.3f} ± {stats['accuracy_std']:.3f}"],
            ['Precision', f"{stats['precision_mean']:.3f} ± {stats['precision_std']:.3f}"],
            ['Recall', f"{stats['recall_mean']:.3f} ± {stats['recall_std']:.3f}"],
            ['F1-Score', f"{stats['f1_mean']:.3f} ± {stats['f1_std']:.3f}"],
            ['AUC-ROC', f"{stats['auc_roc_mean']:.3f} ± {stats['auc_roc_std']:.3f}"],
            ['AUC-PR', f"{stats['auc_pr_mean']:.3f} ± {stats['auc_pr_std']:.3f}"],
            ['MCC', f"{stats['mcc_mean']:.3f} ± {stats['mcc_std']:.3f}"]
        ]
        
        table = ax.table(cellText=table_data,
                        colLabels=['Metric', 'Mean ± Std'],
                        cellLoc='center',
                        loc='center',
                        colWidths=[0.4, 0.6])
        
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1, 2)
        
        # Styling della tabella
        for i in range(len(table_data) + 1):
            for j in range(2):
                if i == 0:  # Header
                    table[(i, j)].set_facecolor(self.colors['primary'])
                    table[(i, j)].set_text_props(weight='bold', color='white')
                else:
                    if i % 2 == 0:
                        table[(i, j)].set_facecolor(self.colors['light'])
        
        ax.set_title('Performance Summary', fontsize=14, fontweight='bold')
        ax.axis('off')
    
    def export_results_to_excel(self, filename='deeptest_results.xlsx'):
        """
        Esporta tutti i risultati in Excel
        """
        if not self.results:
            print("Esegui prima run_comprehensive_test()")
            return
        
        with pd.ExcelWriter(filename, engine='openpyxl') as writer:
            # Sheet 1: Risultati individuali
            self.results['individual_results'].to_excel(
                writer, sheet_name='Individual_Results', index=False)
            
            # Sheet 2: Statistiche aggregate
            agg_df = pd.DataFrame.from_dict(
                self.results['aggregate_stats'], orient='index', 
                columns=['Value']).reset_index()
            agg_df.rename(columns={'index': 'Metric'}, inplace=True)
            agg_df.to_excel(writer, sheet_name='Aggregate_Stats', index=False)
            
            # Sheet 3: Predizioni complete
            pred_df = pd.DataFrame({
                'true_labels': self.results['all_true_labels'],
                'predictions': self.results['all_predictions'], 
                'probabilities': self.results['all_probabilities']
            })
            pred_df.to_excel(writer, sheet_name='All_Predictions', index=False)
        
        print(f"✅ Risultati esportati in: {filename}")
    
    def get_executive_summary(self):
        """
        Genera un riassunto esecutivo per stakeholders non tecnici
        """
        if not self.results:
            print("Esegui prima run_comprehensive_test()")
            return
        
        stats = self.results['aggregate_stats']
        
        summary = f"""
╔═══════════════════════════════════════════════════════════════════════════════╗
║                           🎯 EXECUTIVE SUMMARY                                ║
║                         TabNet Model Performance                              ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║                                                                               ║
║ OVERALL PERFORMANCE:                                                       ║
║    • Model Accuracy: {stats['accuracy_mean']:.1%} (±{stats['accuracy_std']:.1%})                                 ║
║    • Prediction Reliability: {stats['auc_roc_mean']:.1%} (AUC Score)                         ║
║    • Model Consistency: {stats['accuracy_std']/stats['accuracy_mean']:.1%} variation across tests                     ║
║                                                                               ║
║ BUSINESS IMPACT:                                                           ║
║    • Correctly identifies {stats['recall_mean']:.1%} of positive cases                       ║
║    • {stats['precision_mean']:.1%} precision - low false alarms                              ║
║    • Balanced performance across {self.results['n_splits']} independent tests                         ║
║                                                                               ║
║ KEY STRENGTHS:                                                             ║
║    • {'High' if stats['accuracy_mean'] > 0.85 else 'Good' if stats['accuracy_mean'] > 0.75 else 'Moderate'} accuracy rate                                                   ║
║    • {'Excellent' if stats['auc_roc_mean'] > 0.9 else 'Good' if stats['auc_roc_mean'] > 0.8 else 'Fair'} discrimination capability                                     ║
║    • {'Very stable' if stats['accuracy_std'] < 0.02 else 'Stable' if stats['accuracy_std'] < 0.05 else 'Variable'} performance                                                ║
║                                                                               ║
║ RECOMMENDATION: {'DEPLOY' if stats['accuracy_mean'] > 0.8 and stats['auc_roc_mean'] > 0.8 else 'REVIEW'}                                                ║
╚═══════════════════════════════════════════════════════════════════════════════╝
        """
        
        print(summary)
        return summary


In [ ]:
test_data_with_features_df = pd.read_parquet(r"E:/data/test_df_enh_encoder_1M.parquet")
#test_labels_clean_df = pd.read_csv(r"C:/Users/atogni/Desktop/rongowai/temp_data/geoq/test_labels_extracted_old_enc.csv")
#test_labels_clean = #test_labels_clean_df['0'].values

In [ ]:

# Inizializza DeepTest
deep_tester = DeepTest(
    model_path="C:\\Users\\atogni\\Desktop\\rongowai\\geo-k-compression_model\\tabnet_no_q_binary_classifier_20250920_114546.zip",
    test_data=pd.DataFrame(full_data_test),
    test_labels=pd.DataFrame(full_labels_test['0']).values()
)

# Carica il modello (dovrai passare la tua classe TabNetBinaryClassifier)
deep_tester.load_model(TabNetBinaryClassifier)

# Esegui test completo
results = deep_tester.run_comprehensive_test(n_splits=10)

# Genera report
deep_tester.print_detailed_report()
deep_tester.get_executive_summary()

# Crea visualizzazioni
deep_tester.create_professional_visualizations(save_path="model_performance.png")

# Esporta risultati
deep_tester.export_results_to_excel("deeptest_results.xlsx")